In [1]:
!nvidia-smi

Fri Oct  2 10:33:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:01:00.0 Off |                    0 |
| N/A   40C    P0             71W /  500W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import os
os.environ["HF_HUB_CACHE"] = f"/scratch/{os.environ['USER']}/hf_cache"

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/etc/python/sitecustomize.py:117: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  mod = _original_import(name, globals, locals, fromlist, level)


In [4]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="cuda",
    attn_implementation="eager",
)

Loading weights: 100%|██████████| 291/291 [00:03<00:00, 76.59it/s]


In [5]:
messages = [
    {"role": "user", "content": "Explain what is per-token quantization in terms of LLM models."},
]

In [6]:
inputs = tok.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to("cuda")

In [7]:
out = model.generate(**inputs, max_new_tokens=100)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [8]:
prompt_len = inputs["input_ids"].shape[-1]

In [9]:
tok.decode(out[0][prompt_len:], skip_special_tokens=True)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


"Per-token quantization is a technique used in Large Language Models (LLMs) to reduce the precision of the model's weights and activations from 32-bit floating-point numbers to 8-bit or 16-bit integers. This is done on a per-token basis, meaning that each token (or input sequence) is processed separately, and the quantization is applied independently for each token.\n\nIn traditional quantization methods, the entire model is quantized, which can lead to a loss of accuracy. Per"

In [10]:
print(f"{torch.cuda.memory_allocated() / 1e9:.1f} GB on GPU")

16.1 GB on GPU
